In [ ]:
!pip install --upgrade --force-reinstall sympy torch torchvision

In [15]:
import os

for root, dirs, files in os.walk('/kaggle/input'):
    if root.count(os.sep) <= 4:
        print("DIR:", root)
    for f in files:
        if f.endswith('.csv'):
            print("CSV:", os.path.join(root, f))

for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.pth'):
            print("PTH:", os.path.join(root, f))


DIR: /kaggle/input
DIR: /kaggle/input/notebooks
DIR: /kaggle/input/notebooks/ingusterbets
DIR: /kaggle/input/datasets
DIR: /kaggle/input/datasets/nih-chest-xrays
CSV: /kaggle/input/datasets/nih-chest-xrays/data/BBox_List_2017.csv
CSV: /kaggle/input/datasets/nih-chest-xrays/data/Data_Entry_2017.csv
DIR: /kaggle/input/datasets/anuragsanadi
CSV: /kaggle/input/datasets/anuragsanadi/train-small-2-csv/train_small 2.csv
PTH: /kaggle/input/datasets/anuragsanadi/resnet18-chest-xray-weights-pth/resnet18_chest_xray_weights.pth


In [5]:
import os
import shutil

src = "/kaggle/input/datasets/anuragsanadi/resnet18-chest-xray-weights-pth/resnet18_chest_xray_weights.pth"
dst = "/kaggle/working/resnet18_chest_xray_weights.pth"

shutil.copy(src, dst)
print("Copied successfully:", os.path.exists(dst))

Copied successfully: True


In [7]:
import os
import pandas as pd

# 1. Define the folder path as a string
files_dir = '/kaggle/input/datasets/anuragsanadi/train-small-2-csv/'

# 2. Print all files present in this folder
files = os.listdir(files_dir)
print("Files in folder:", files)

# 3. Build the full file path using the filename string and import with pandas
training_set_csv_path = os.path.join(files_dir, 'train_small_2.csv')
training_image_paths = pd.read_csv(training_set_csv_path)

test_set_csv_path = os.path.join(files_dir, "test_small_2.csv")
testing_image_paths = pd.read_csv(test_set_csv_path)

# Preview the loaded DataFrame
training_image_paths.head()
testing_image_paths.head()


Files in folder: ['test_small_2.csv', 'train_small_2.csv']


,Image Index,Patient ID,View Position,image_path,Atelectasis,Cardiomegaly,Effusion,Infiltration,Mass,Nodule,Pneumonia,Pneumothorax,Consolidation,Edema,Emphysema,Fibrosis,Pleural_Thickening,Hernia
0,00000583_001.png,583,PA,/Users/eric/Downloads/ChestXray-NIHCC/extracte...,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,00000583_002.png,583,AP,/Users/eric/Downloads/ChestXray-NIHCC/extracte...,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,00000583_003.png,583,AP,/Users/eric/Downloads/ChestXray-NIHCC/extracte...,1,0,0,0,0,0,0,0,0,0,0,0,0,0
3,00000583_004.png,583,PA,/Users/eric/Downloads/ChestXray-NIHCC/extracte...,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,00000583_005.png,583,AP,/Users/eric/Downloads/ChestXray-NIHCC/extracte...,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
!pip install --force-reinstall --no-cache-dir numpy

In [ ]:
# Completely wipe any conflicting versions
# !pip uninstall -y torch torchvision torchtext

# Reinstall the latest versions directly from PyTorch's official source with modern CUDA support
# !pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu126

In [8]:
import os
import kagglehub
import pandas as pd
from glob import glob

path = kagglehub.dataset_download("nih-chest-xrays/data")

# 1. Load Data_Entry_2017.csv (without nrows=2000 so no matching filenames get cut off)
dataset = pd.read_csv(os.path.join(path, "Data_Entry_2017.csv"))

# 2. Index available PNG files on disk (use "images_*" to search all image folders, or "images_001" for just folder 1)
target_folder = os.path.join(path, "images_*", "images", "*.png")
all_image_paths = glob(target_folder)
image_path_dict = {os.path.basename(p): p for p in all_image_paths}

# 3. Pick rows in Data_Entry_2017.csv whose 'Image Index' is in train_small 2.csv AND exists on disk
df = dataset[
    dataset["Image Index"].isin(training_image_paths["Image Index"]) &
    dataset["Image Index"].isin(image_path_dict.keys())
].reset_index(drop=True)

print(f"Rows remaining: {len(df)}")
df.head()

Rows remaining: 7263


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000035_000.png,No Finding,0,35,79,M,PA,2754,2873,0.143000,0.143000,NaN
1,00000035_001.png,No Finding,1,35,80,M,PA,2718,2973,0.143000,0.143000,NaN
2,00000036_000.png,Effusion|Infiltration,0,36,81,M,PA,2992,2991,0.143000,0.143000,NaN
3,00000043_000.png,Infiltration,0,43,66,M,PA,2794,2991,0.143000,0.143000,NaN
4,00000044_000.png,Consolidation|Effusion|Infiltration,0,44,79,M,PA,2010,2021,0.194311,0.194311,NaN


In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os

from glob import glob
from PIL import Image
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub

path = kagglehub.dataset_download("nih-chest-xrays/data")

print("Path to dataset files:", path)

# for filename in os.listdir(path):
#     print(filename)

# print(os.getcwd())
path = kagglehub.dataset_download("nih-chest-xrays/data")

# 1. Load Data_Entry_2017.csv (without nrows=2000 so no matching filenames get cut off)
dataset = pd.read_csv(os.path.join(path, "Data_Entry_2017.csv"))

# 2. Index available PNG files on disk (use "images_*" to search all image folders, or "images_001" for just folder 1)
target_folder = os.path.join(path, "images_*", "images", "*.png")
all_image_paths = glob(target_folder)
image_path_dict = {os.path.basename(p): p for p in all_image_paths}

# 3. Pick rows in Data_Entry_2017.csv whose 'Image Index' is in train_small 2.csv AND exists on disk
df = dataset[
    dataset["Image Index"].isin(training_image_paths["Image Index"]) &
    dataset["Image Index"].isin(image_path_dict.keys())
].reset_index(drop=True)

print(f"Rows remaining: {len(df)}")
df.head()

# X = df[['Image Index']]
# y = df['Finding Labels']

# # Pass the Patient_id column so the splitter keeps patients intact
# gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# for train_idx, test_idx in gss.split(X, y, groups=df['Patient ID']):
#     train_set = df.iloc[train_idx]
#     test_set = df.iloc[test_idx]

train_set = df
test_set = dataset[
    dataset["Image Index"].isin(testing_image_paths["Image Index"]) &
    dataset["Image Index"].isin(image_path_dict.keys())
].reset_index(drop=True)

# print("Train set: \n", train_set)
# print("Test set: \n",  test_set)

class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, image_path_dict, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_path_dict = image_path_dict
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row["Image Index"]

        img_path = self.image_path_dict.get(img_name)
        if img_path is None:
            raise FileNotFoundError(
                f"Could not find file for image index: {img_name}"
            )

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label_str = row["Finding Labels"]
        label = 0.0 if "No Finding" in label_str else 1.0
        label_tensor = torch.tensor(label, dtype=torch.float32)

        return image, label_tensor



# 8. Define standard CNN image preprocessing transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
    ),
])

train_dataset = ChestXrayDataset(train_set, image_path_dict, transform=transform)
test_dataset = ChestXrayDataset(test_set, image_path_dict, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 11. Sanity check a single batch
images, labels = next(iter(train_loader))
print("Batch image tensor shape:", images.shape)  # Expected: torch.Size([32, 3, 224, 224])
print("Batch labels (first 5):", labels[:5])

Path to dataset files: /kaggle/input/datasets/nih-chest-xrays/data
Rows remaining: 7263
Batch image tensor shape: torch.Size([32, 3, 224, 224])
Batch labels (first 5): tensor([1., 0., 1., 1., 1.])


In [6]:
# Code for the Conv network and training

# 11. Define Neural Network (2 Convolutional Layers + 1 ANN Classifier)
class SimpleCNN(nn.Module):

  def __init__(self):
    super(SimpleCNN, self).__init__()
    # Convolutional Layer 1
    self.conv1 = nn.Conv2d(
        in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1
    )
    # Convolutional Layer 2
    self.conv2 = nn.Conv2d(
        in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1
    )
    self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    # ANN / Fully Connected Layer (Input dimensions: 32 channels * 56 * 56 features)
    self.fc1 = nn.Linear(32 * 56 * 56, 1)

  def forward(self, x):
    # Pass through Conv Layer 1 -> ReLU -> MaxPool
    x = self.pool(F.relu(self.conv1(x)))
    # Pass through Conv Layer 2 -> ReLU -> MaxPool
    x = self.pool(F.relu(self.conv2(x)))

    # Flatten feature maps into a 1D vector for the ANN layer
    x = x.view(x.size(0), -1)

    # Pass through ANN linear layer with a Sigmoid activation for binary output
    x = torch.sigmoid(self.fc1(x))
    return x


# # 12. Initialize Model, Loss Function, and Optimizer
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")

# model = SimpleCNN().to(device)
# criterion = (
#     nn.BCELoss()
# )  # Binary Cross-Entropy Loss for binary classification
# optimizer = optim.Adam(model.parameters(), lr=0.001)

# Resnet-18 model with pretrained weights
import torchvision.models as models


# 11. Define Neural Network using ResNet18
class ResNet18Binary(nn.Module):

  def __init__(self, pretrained=True):
    super(ResNet18Binary, self).__init__()
    # Load ResNet18 (using weights='DEFAULT' or pretrained=True depending on torchvision version)
    try:
      weights = models.ResNet18_Weights.DEFAULT if pretrained else None
      self.resnet = models.resnet18(weights=weights)
    except AttributeError:
      # Fallback for older torchvision versions
      self.resnet = models.resnet18(pretrained=pretrained)

    # ResNet18 outputs 512 features from its final average pooling layer before the fc layer.
    # Replace the default 1000-class classifier with a 1-output linear layer for binary classification.
    num_features = self.resnet.fc.in_features
    self.resnet.fc = nn.Linear(num_features, 1)

  def forward(self, x):
    # Pass through ResNet layers
    x = self.resnet(x)
    # Apply sigmoid activation to get a probability between 0 and 1
    x = torch.sigmoid(x)
    return x

# 12. Initialize Model, Loss Function, and Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize ResNet18 binary model
model = ResNet18Binary(pretrained=True).to(device)

# to train Resnet-18 model from scratch
# model = ResNet18Binary(pretrained=False).to(device)

criterion = nn.BCELoss()  # Binary Cross-Entropy Loss
optimizer = optim.Adam(
    model.parameters(), lr=0.0001
)  # Lower learning rate (e.g., 1e-4) is often better for pre-trained weights

# 13. Training Loop
num_epochs = 15
print("Starting training loop...")

for epoch in range(num_epochs):
  model.train()
  running_loss = 0.0

  for images, labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)

    # Zero out gradients from previous step
    optimizer.zero_grad()

    # Forward pass
    outputs = model(images).squeeze(1)

    # Compute loss
    loss = criterion(outputs, labels)

    # Backward pass and optimize
    loss.backward()
    optimizer.step()

    running_loss += loss.item() * images.size(0)

  epoch_loss = running_loss / len(train_dataset)
  print(f"Epoch [{epoch+1}/{num_epochs}], Training Loss: {epoch_loss:.4f}")



print("Training complete!")

save_path = "/kaggle/working/resnet18_chest_xray_weights.pth"
torch.save(model.state_dict(), save_path)
print(f"Model weights saved to {save_path}")

Using device: cuda
Starting training loop...
Epoch [1/15], Training Loss: 0.6256
Epoch [2/15], Training Loss: 0.4964
Epoch [3/15], Training Loss: 0.3260
Epoch [4/15], Training Loss: 0.1447
Epoch [5/15], Training Loss: 0.0969
Epoch [6/15], Training Loss: 0.0724
Epoch [7/15], Training Loss: 0.0637
Epoch [8/15], Training Loss: 0.0821
Epoch [9/15], Training Loss: 0.0600
Epoch [10/15], Training Loss: 0.0468
Epoch [11/15], Training Loss: 0.0500
Epoch [12/15], Training Loss: 0.0234
Epoch [13/15], Training Loss: 0.0209
Epoch [14/15], Training Loss: 0.0187
Epoch [15/15], Training Loss: 0.0328
Training complete!
Model weights saved to /kaggle/working/resnet18_chest_xray_weights.pth


In [10]:
import torchvision.models as models
from glob import glob
from PIL import Image
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
# 11. Define Neural Network using ResNet18
class ResNet18Binary(nn.Module):

  def __init__(self, pretrained=True):
    super(ResNet18Binary, self).__init__()
    # Load ResNet18 (using weights='DEFAULT' or pretrained=True depending on torchvision version)
    try:
      weights = models.ResNet18_Weights.DEFAULT if pretrained else None
      self.resnet = models.resnet18(weights=weights)
    except AttributeError:
      # Fallback for older torchvision versions
      self.resnet = models.resnet18(pretrained=pretrained)

    # ResNet18 outputs 512 features from its final average pooling layer before the fc layer.
    # Replace the default 1000-class classifier with a 1-output linear layer for binary classification.
    num_features = self.resnet.fc.in_features
    self.resnet.fc = nn.Linear(num_features, 1)

  def forward(self, x):
    # Pass through ResNet layers
    x = self.resnet(x)
    # Apply sigmoid activation to get a probability between 0 and 1
    x = torch.sigmoid(x)
    return x

# 12. Initialize Model, Loss Function, and Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize ResNet18 binary model
model = ResNet18Binary(pretrained=True).to(device)

# to train Resnet-18 model from scratch
# model = ResNet18Binary(pretrained=False).to(device)

criterion = nn.BCELoss()  # Binary Cross-Entropy Loss
optimizer = optim.Adam(
    model.parameters(), lr=0.001
)  # Lower learning rate (e.g., 1e-4) is often better for pre-trained weights


Using device: cuda


In [13]:
# 1. Re-instantiate the model architecture
loaded_model = ResNet18Binary(pretrained=False).to(device)

# 2. Load the saved state_dict
loaded_model.load_state_dict(
    torch.load("/kaggle/working/resnet18_chest_xray_weights.pth", map_location=device, weights_only=True)
)

# 3. Set to evaluation mode before running predictions
loaded_model.eval()

ResNet18Binary(
  (resnet): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, tra

In [11]:
# 1. Re-instantiate the model architecture
loaded_model = ResNet18Binary(pretrained=False).to(device)

# 2. Load the saved state_dict
loaded_model.load_state_dict(
    torch.load("/kaggle/working/resnet18_chest_xray_weights.pth", map_location=device, weights_only=True)
)

# 3. Set to evaluation mode before running predictions
loaded_model.eval()

# 14. Testing and Evaluation Phase
test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():  # Disable gradient calculations during evaluation
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass using loaded_model
        outputs = loaded_model(images).squeeze(1)

        # Compute loss
        loss = criterion(outputs, labels)
        test_loss += loss.item() * images.size(0)

        # Calculate accuracy (threshold outputs at 0.5 for binary classification)
        predicted = (outputs >= 0.5).float()
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

# Compute final average loss and accuracy
avg_test_loss = test_loss / len(test_dataset)
test_accuracy = correct / total

print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}% ({correct}/{total} correct)")

Test Loss: 0.6509
Test Accuracy: 64.70% (3725/5757 correct)


In [3]:
!python /kaggle/input/datasets/paknathan450/data255-files/train.py --train-csv /kaggle/input/datasets/paknathan450/data255-files/train_50.csv --val-csv /kaggle/input/datasets/paknathan450/data255-files/val_small.csv \
      --test-csv /kaggle/input/datasets/paknathan450/data255-files/test_small.csv --image-root /kaggle/input/datasets/nih-chest-xrays \
      --model resnet18 --fraction 50 --epochs 10 --output-dir runs/resnet18_full --pretrained

resolved train: 3632 images
resolved val: 2872 images
resolved test: 5757 images
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 232MB/s]
{"epoch": 1, "train_loss": 1.2823092953224013, "val_loss": 1.3984339051259926, "val_auc": 0.6640494401732201, "lr": 9.755282581475769e-05, "epoch_time_sec": 78.8}
{"epoch": 2, "train_loss": 0.9890464959165598, "val_loss": 1.4123654226074644, "val_auc": 0.6890264240818247, "lr": 9.045084971874737e-05, "epoch_time_sec": 59.5}
{"epoch": 3, "train_loss": 0.8122985699103267, "val_loss": 1.4986118310673324, "val_auc": 0.6924092218680091, "lr": 7.938926261462366e-05, "epoch_time_sec": 59.4}
{"epoch": 4, "train_loss": 0.6586785799606256, "val_loss": 1.6562880998202354, "val_auc": 0.7055468397105125, "lr": 6.545084971874737e-05, "epoch_time_sec": 59.8}
{"epoch": 5, "train_loss": 0.5364542298905125, "val_l

In [13]:
import shutil

shutil.make_archive('/kaggle/working/working_archive', 'zip', '/kaggle/working')


'/kaggle/working/working_archive.zip'

In [ ]:
import os
import zipfile
from IPython.display import FileLink

zip_path = '/kaggle/working/working_archive.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
  for root, dirs, files in os.walk('/kaggle/working'):
    for file in files:
      full_path = os.path.join(root, file)
      # Skip the archive itself so it doesn't loop
      if full_path != zip_path:
        rel_path = os.path.relpath(full_path, '/kaggle/working')
        zipf.write(full_path, rel_path)

FileLink('working_archive.zip')


In [12]:
!python /kaggle/input/datasets/paknathan450/data255-files/train.py --train-csv /kaggle/input/datasets/paknathan450/data255-files/train_25.csv --val-csv /kaggle/input/datasets/paknathan450/data255-files/val_small.csv \
      --test-csv /kaggle/input/datasets/paknathan450/data255-files/test_small.csv --image-root /kaggle/input/datasets/nih-chest-xrays \
      --model resnet18 --fraction 25 --epochs 10 --output-dir runs/resnet18_full --pretrained

resolved train: 1828 images
resolved val: 2872 images
resolved test: 5757 images
{"epoch": 1, "train_loss": 1.307694298694118, "val_loss": 1.5346703642258073, "val_auc": 0.6081053095374396, "lr": 9.755282581475769e-05, "epoch_time_sec": 41.9}
{"epoch": 2, "train_loss": 0.9661807956528612, "val_loss": 1.527895924108606, "val_auc": 0.6520803052860132, "lr": 9.045084971874737e-05, "epoch_time_sec": 41.5}
{"epoch": 3, "train_loss": 0.8101067689106813, "val_loss": 1.5854521368871493, "val_auc": 0.6563506595492578, "lr": 7.938926261462366e-05, "epoch_time_sec": 41.7}
{"epoch": 4, "train_loss": 0.6494605499492954, "val_loss": 1.7783861575352449, "val_auc": 0.6464624261231201, "lr": 6.545084971874737e-05, "epoch_time_sec": 42.0}
{"epoch": 5, "train_loss": 0.5432525921441794, "val_loss": 1.8812695426197106, "val_auc": 0.6533277098378102, "lr": 4.9999999999999996e-05, "epoch_time_sec": 41.9}
{"epoch": 6, "train_loss": 0.4638622570090012, "val_loss": 2.0067298505299602, "val_auc": 0.6481609935922

In [11]:
!rm -rf /kaggle/working/*
